# Quantum-Hybrid CVRP Solver — qCourier Yale Hackathon 2026

**Hybrid structure:**
- **Classical** (from `classical_solver.py`): distance matrix → minimum vehicles → sweep clustering → vehicle assignment
- **Quantum** (QAOA via Qiskit): replaces `build_route` — finds the optimal visit order within each group

In [1]:
# !pip install qiskit qiskit-aer scipy numpy

In [2]:
import os
import time
import numpy as np
from itertools import permutations
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_aer import AerSimulator

from classical_solver import (
    encode_locations, build_distance_matrix, compute_num_groups,
    cluster_houses, assign_vehicles, total_distance
)

simulator = AerSimulator()

---
## Quantum Component — QAOA `build_route`

Given a group of `k` customers, find the shortest round-trip route depot → ... → depot.

### Step 1 — Formulate as QUBO

Define binary variables `x[pos][c] = 1` if customer `c` is at position `pos` in the route (`k²` variables total).

- **Constraint 1:** each position has exactly one customer
- **Constraint 2:** each customer appears exactly once
- **Objective:** minimize total distance

In [3]:
def build_qubo(group, dist_matrix, penalty=None):
    k = len(group)
    n = k * k
    Q = np.zeros((n, n))

    if penalty is None:
        penalty = 2 * sum(
            dist_matrix[(a, b)]
            for a in [0] + group
            for b in [0] + group
            if a != b
        )

    def v(pos, c):
        return pos * k + c

    # Constraint 1: each position has exactly one customer
    for pos in range(k):
        for c in range(k):
            Q[v(pos,c), v(pos,c)] -= penalty
        for c1 in range(k):
            for c2 in range(c1+1, k):
                Q[v(pos,c1), v(pos,c2)] += 2 * penalty

    # Constraint 2: each customer appears exactly once
    for c in range(k):
        for pos in range(k):
            Q[v(pos,c), v(pos,c)] -= penalty
        for p1 in range(k):
            for p2 in range(p1+1, k):
                Q[v(p1,c), v(p2,c)] += 2 * penalty

    # Objective: depot -> first customer
    for c, cust in enumerate(group):
        Q[v(0,c), v(0,c)] += dist_matrix[(0, cust)]

    # Objective: customer -> customer (skip same customer)
    for pos in range(k-1):
        for c1, cu1 in enumerate(group):
            for c2, cu2 in enumerate(group):
                if cu1 == cu2:
                    continue
                d = dist_matrix[(cu1, cu2)]
                i, j = v(pos, c1), v(pos+1, c2)
                if i <= j:
                    Q[i, j] += d
                else:
                    Q[j, i] += d

    # Objective: last customer -> depot
    for c, cust in enumerate(group):
        Q[v(k-1,c), v(k-1,c)] += dist_matrix[(cust, 0)]

    return Q

### Step 2 — Convert QUBO to Ising

Substitute `x_i = (1 - z_i) / 2` to get the Ising Hamiltonian `H = Σ h_i Z_i + Σ J_ij Z_i Z_j` that QAOA minimizes.

In [4]:
def qubo_to_ising(Q):
    n = Q.shape[0]
    h = np.zeros(n)
    J = {}
    offset = 0.0
    for i in range(n):
        h[i] -= Q[i,i] / 2
        offset += Q[i,i] / 2
        for j in range(i+1, n):
            if abs(Q[i,j]) > 1e-10:
                J[(i,j)] = Q[i,j] / 4
                h[i] -= Q[i,j] / 4
                h[j] -= Q[i,j] / 4
                offset += Q[i,j] / 4
    return h, J, offset

### Step 3 — Build QAOA Circuit in Qiskit

- Start in uniform superposition `|+⟩^n`
- Alternate `p` layers of cost unitary (ZZ rotations) and mixer (X rotations)
- Parameters `γ` (cost) and `β` (mixer) are optimized classically

In [5]:
def build_qaoa_circuit(n_qubits, h, J, p=1):
    gammas = ParameterVector('γ', p)
    betas  = ParameterVector('β', p)

    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))  # uniform superposition

    for layer in range(p):
        # Cost unitary
        for i in range(n_qubits):
            if abs(h[i]) > 1e-10:
                qc.rz(2 * gammas[layer] * h[i], i)
        for (i, j), Jij in J.items():
            if abs(Jij) > 1e-10:
                qc.cx(i, j)
                qc.rz(2 * gammas[layer] * Jij, j)
                qc.cx(i, j)
        # Mixer unitary
        for i in range(n_qubits):
            qc.rx(2 * betas[layer], i)

    qc.measure_all()
    return qc, list(gammas), list(betas)

### Step 4 — Optimize Parameters and Decode Route

In [6]:
def run_circuit(qc, params, param_list, shots=512):
    """Bind parameters and run circuit on Qiskit Aer simulator."""
    bound = qc.assign_parameters({param_list[i]: params[i] for i in range(len(params))})
    job = simulator.run(bound, shots=shots)
    return job.result().get_counts()


def optimize_qaoa(qc, gammas, betas, h, J, p=1, shots=512, maxiter=100):
    """COBYLA loop: minimize Ising expectation value over γ, β parameters."""
    all_params = list(gammas) + list(betas)
    n = qc.num_qubits

    def expectation(params):
        counts = run_circuit(qc, params, all_params, shots)
        energy, total = 0.0, sum(counts.values())
        for bitstring, count in counts.items():
            z = [1 - 2*int(b) for b in reversed(bitstring)]
            e  = sum(h[i] * z[i] for i in range(n))
            e += sum(Jij * z[i] * z[j] for (i,j), Jij in J.items())
            energy += count * e
        return energy / total

    best_params, best_val = None, float('inf')
    for _ in range(3):  # 3 random restarts
        x0  = np.random.uniform(0, 2*np.pi, 2*p)
        res = minimize(expectation, x0, method='COBYLA', options={'maxiter': maxiter})
        if res.fun < best_val:
            best_val, best_params = res.fun, res.x
    return best_params


def decode_route(qc, params, all_params, group, dist_matrix, shots=1024):
    """Sample the optimized circuit; decode the most-frequent valid permutation into a route."""
    counts = run_circuit(qc, params, all_params, shots)
    k = len(group)
    best_route, best_dist = None, float('inf')

    for bitstring in sorted(counts, key=lambda b: -counts[b]):
        bits = [int(b) for b in reversed(bitstring)]
        x = np.array(bits).reshape(k, k)
        if not (np.all(x.sum(axis=1) == 1) and np.all(x.sum(axis=0) == 1)):
            continue
        route = [0] + [group[int(np.argmax(x[pos]))] for pos in range(k)] + [0]
        d = sum(dist_matrix[(route[i], route[i+1])] for i in range(len(route)-1))
        if d < best_dist:
            best_dist, best_route = d, route

    if best_route is None:  # fallback if QAOA found no valid permutation
        for perm in permutations(group):
            route = [0] + list(perm) + [0]
            d = sum(dist_matrix[(route[i], route[i+1])] for i in range(len(route)-1))
            if d < best_dist:
                best_dist, best_route = d, route

    return best_route

### QAOA `build_route` — drops into the classical skeleton

In [7]:
def build_route_qaoa(group, dist_matrix, p=1, shots=512, maxiter=100):
    """QAOA replacement for classical build_route."""
    if len(group) == 1:
        return [0, group[0], 0]

    Q            = build_qubo(group, dist_matrix)
    h, J, _      = qubo_to_ising(Q)
    n_qubits     = len(group) ** 2
    qc, gammas, betas = build_qaoa_circuit(n_qubits, h, J, p=p)
    all_params   = list(gammas) + list(betas)
    opt_params   = optimize_qaoa(qc, gammas, betas, h, J, p=p, shots=shots, maxiter=maxiter)
    return decode_route(qc, opt_params, all_params, group, dist_matrix, shots=shots)

---
## Full Hybrid Solver

Identical to `solve_cvrp` in `classical_solver.py`, except `build_route` is swapped for `build_route_qaoa`.

In [8]:
def solve_cvrp_hybrid(depot, customers, N, C, p=1, shots=512, maxiter=100, verbose=True):
    locations   = encode_locations(depot, customers)
    dist_matrix = build_distance_matrix(locations)
    H = len(customers)
    G = compute_num_groups(H, C, N)

    if verbose:
        print(f"  G = min(ceil({H}/{C}), {N}) = {G} vehicle(s)")

    groups     = cluster_houses(locations, dist_matrix, G, C)
    assignment = assign_vehicles(groups, N)

    if verbose:
        print(f"  Groups: {groups}")

    routes      = []
    resource_log = []

    for v in sorted(assignment):
        for group in assignment[v]:
            k        = len(group)
            n_qubits = k * k

            if verbose:
                print(f"  [QAOA] vehicle {v}, group {group}  →  {n_qubits} qubits")

            t0    = time.time()
            route = build_route_qaoa(group, dist_matrix, p=p, shots=shots, maxiter=maxiter)
            elapsed = round(time.time() - t0, 2)

            # Count gates on a sample bound circuit
            Q           = build_qubo(group, dist_matrix)
            h, J, _     = qubo_to_ising(Q)
            qc, _, _    = build_qaoa_circuit(n_qubits, h, J, p=p)
            gate_count  = sum(qc.decompose().count_ops().values())

            routes.append((v, route))
            resource_log.append({"qubits": n_qubits, "gates": gate_count, "time_s": elapsed})

            if verbose:
                d = sum(dist_matrix[(route[i], route[i+1])] for i in range(len(route)-1))
                print(f"    route: {' -> '.join(str(x) for x in route)}  dist={d:.4f}  time={elapsed}s")

    dist = total_distance(routes, dist_matrix)
    if verbose:
        print(f"\n  Total distance: {dist:.4f}")

    return routes, dist, resource_log

---
## Run All Instances

In [9]:
depot = (0, 0)

instances = {
    1: {"customers": [(-2,2),(-5,8),(2,3)],                                                                               "N": 2, "C": 5},
    2: {"customers": [(-2,2),(-5,8),(2,3)],                                                                               "N": 2, "C": 2},
    3: {"customers": [(-2,2),(-5,8),(2,3),(5,7),(2,4),(2,-3)],                                                            "N": 3, "C": 2},
    4: {"customers": [(-2,2),(-5,8),(6,3),(4,4),(3,2),(0,2),(-2,3),(-4,3),(2,3),(2,7),(-2,5),(-1,4)], "N": 4, "C": 3},
}

all_results  = {}
resource_table = []

for inst_num, params in instances.items():
    print(f"{'='*55}")
    print(f"Instance {inst_num}  |  N={params['N']}, C={params['C']}")
    print(f"{'='*55}")
    routes, dist, rlog = solve_cvrp_hybrid(
        depot, params["customers"], params["N"], params["C"]
    )
    all_results[inst_num] = {"routes": routes, "total_distance": dist}
    resource_table.append({
        "instance": inst_num,
        "qubits":   max(r["qubits"] for r in rlog),
        "gates":    sum(r["gates"]  for r in rlog),
        "time_s":   round(sum(r["time_s"] for r in rlog), 2)
    })
    print()

Instance 1  |  N=2, C=5
  G = min(ceil(3/5), 2) = 1 vehicle(s)
  Groups: [[3, 2, 1]]
  [QAOA] vehicle 1, group [3, 2, 1]  →  9 qubits


    route: 0 -> 1 -> 2 -> 3 -> 0  dist=21.7445  time=0.3s

  Total distance: 21.7445

Instance 2  |  N=2, C=2
  G = min(ceil(3/2), 2) = 2 vehicle(s)
  Groups: [[2, 1], [3]]
  [QAOA] vehicle 1, group [2, 1]  →  4 qubits
    route: 0 -> 1 -> 2 -> 0  dist=18.9706  time=0.08s
  [QAOA] vehicle 2, group [3]  →  1 qubits
    route: 0 -> 3 -> 0  dist=7.2111  time=0.0s

  Total distance: 26.1817

Instance 3  |  N=3, C=2
  G = min(ceil(6/2), 3) = 3 vehicle(s)
  Groups: [[6, 4], [3, 5], [2, 1]]
  [QAOA] vehicle 1, group [6, 4]  →  4 qubits
    route: 0 -> 4 -> 6 -> 0  dist=22.6482  time=0.08s
  [QAOA] vehicle 2, group [3, 5]  →  4 qubits


    route: 0 -> 3 -> 5 -> 0  dist=9.0777  time=0.08s
  [QAOA] vehicle 3, group [2, 1]  →  4 qubits
    route: 0 -> 2 -> 1 -> 0  dist=18.9706  time=0.06s

  Total distance: 50.6965

Instance 4  |  N=4, C=3
  G = min(ceil(12/3), 4) = 4 vehicle(s)
  Groups: [[3, 5, 4], [9, 10, 6], [12, 11, 2], [7, 1, 8]]
  [QAOA] vehicle 1, group [3, 5, 4]  →  9 qubits


    route: 0 -> 3 -> 4 -> 5 -> 0  dist=14.7859  time=0.28s
  [QAOA] vehicle 2, group [9, 10, 6]  →  9 qubits


    route: 0 -> 9 -> 10 -> 6 -> 0  dist=14.9907  time=0.25s
  [QAOA] vehicle 3, group [12, 11, 2]  →  9 qubits


    route: 0 -> 12 -> 2 -> 11 -> 0  dist=19.4078  time=0.29s
  [QAOA] vehicle 4, group [7, 1, 8]  →  9 qubits


    route: 0 -> 1 -> 8 -> 7 -> 0  dist=10.6700  time=0.25s

  Total distance: 59.8544



## Resource Usage Table

In [10]:
print(f"{'Instance':<12} {'# Qubits':<12} {'# Gates':<12} {'Time (s)':<12}")
print("-" * 48)
for row in resource_table:
    print(f"{row['instance']:<12} {row['qubits']:<12} {row['gates']:<12} {row['time_s']:<12}")

Instance     # Qubits     # Gates      Time (s)    
------------------------------------------------
1            9            127          0.3         
2            4            40           0.08        
3            4            105          0.22        
4            9            508          1.07        


## Write Solution Files

In [11]:
os.makedirs("solutions_quantum", exist_ok=True)

for inst_num, res in all_results.items():
    path = f"solutions_quantum/Instance{inst_num}.txt"
    with open(path, "w") as f:
        for i, (_, route) in enumerate(res["routes"], start=1):
            f.write("r" + str(i) + ": " + ", ".join(str(n) for n in route) + "\n")
    print(f"Written {path}")

Written solutions_quantum/Instance1.txt
Written solutions_quantum/Instance2.txt
Written solutions_quantum/Instance3.txt
Written solutions_quantum/Instance4.txt
